
<a href="https://colab.research.google.com/github/adenikeadewumi/python-ml-WIEOAU/blob/main/11_pandas/exercises/11_solutions.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Solutions — Module 11: Pandas

---

### Exercise 1 — Create DataFrame and compute group averages

**Concept:** `groupby()` splits the data into groups based on a column's values, then `.agg()` applies a function (like 'mean') to each group.

In [ ]:
import pandas as pd
import numpy as np

data = {
    'id':         [1, 2, 3, 4, 5, 6, 7, 8],
    'name':       ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Henry'],
    'department': ['Eng', 'Marketing', 'Eng', 'Finance', 'Marketing', 'Eng', 'Finance', 'Eng'],
    'salary':     [72000, 55000, 68000, 82000, 59000, 65000, 91000, 70000],
    'years_exp':  [4, 7, 3, 12, 5, 2, 15, 6]
}
df = pd.DataFrame(data)
print(df)

print("
Average salary by department:")
avg_salary = df.groupby('department')['salary'].mean().round(0)
print(avg_salary)

### Exercise 2 — Employees above median salary in their department

**Concept:** Use `groupby().transform()` — this applies a function per group but returns a result with the SAME index as the original DataFrame. Perfect for filtering based on group-level statistics.

In [ ]:
# Calculate the median salary for each department
# transform() returns a Series aligned with df's index — one value per row
df['dept_median'] = df.groupby('department')['salary'].transform('median')

# Now filter: keep rows where salary > dept_median
above_median = df[df['salary'] > df['dept_median']]

print("Employees earning above their department's median salary:")
print(above_median[['name', 'department', 'salary', 'dept_median']])

# Why transform() instead of agg()? 
# agg() returns one row per group (compressed)
# transform() returns the same shape as the original df (stretched)
# We need transform() because we want to compare each row to its group's median

**Key distinction:** `.agg()` reduces groups to summary statistics. `.transform()` maps group statistics back to every row in the original DataFrame. Use transform when you want to filter or compare individual rows against their group.

### Exercise 3 — Fill missing values with department median

**Concept:** Introduce NaN values, then use `groupby().transform()` again to compute per-department medians, then fill the nulls.

In [ ]:
import numpy as np

df_missing = df.copy()
# Introduce 3 missing salary values
df_missing.loc[1, 'salary'] = np.nan
df_missing.loc[4, 'salary'] = np.nan
df_missing.loc[6, 'salary'] = np.nan

print("Salaries with missing values:")
print(df_missing[['name', 'department', 'salary']])

print(f"
Missing values: {df_missing['salary'].isna().sum()}")

# Fill with department median using transform
dept_medians = df_missing.groupby('department')['salary'].transform('median')
df_missing['salary'] = df_missing['salary'].fillna(dept_medians)

print("
After filling with department median:")
print(df_missing[['name', 'department', 'salary']])
print(f"Missing values remaining: {df_missing['salary'].isna().sum()}")

### Exercise 4 — Mini EDA Challenge

**Concept:** Exploratory Data Analysis (EDA) is the first thing you do with any new dataset. The goal is to understand the shape, types, quality, and patterns in the data before building any model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

# Using a small built-in dataset to avoid needing internet
# In practice: df = pd.read_csv('your_data.csv')
csv_data = """PassengerId,Survived,Pclass,Name,Sex,Age,Fare
1,0,3,Braund Mr Owen,male,22,7.25
2,1,1,Cumings Mrs John,female,38,71.28
3,1,3,Heikkinen Miss Laina,female,26,7.92
4,1,1,Futrelle Mrs Jacques,female,35,53.1
5,0,3,Allen Mr William,male,35,8.05
6,0,3,Moran Mr James,male,,8.46
7,0,1,McCarthy Mr Timothy,male,54,51.86
8,0,3,Palsson Master Gosta,male,2,21.08
9,1,3,Johnson Mrs Oscar,female,27,11.13
10,1,2,Nasser Mrs Nicholas,female,14,30.07"""

df = pd.read_csv(StringIO(csv_data))

print("=" * 40)
print("STEP 1: Shape and columns")
print("=" * 40)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("Columns:", list(df.columns))

print("
" + "=" * 40)
print("STEP 2: Data types")
print("=" * 40)
print(df.dtypes)

print("
" + "=" * 40)
print("STEP 3: Missing values")
print("=" * 40)
missing = df.isnull().sum()
print(missing[missing > 0])

print("
" + "=" * 40)
print("STEP 4: Numerical summary")
print("=" * 40)
print(df.describe().round(2))

print("
" + "=" * 40)
print("STEP 5: Interesting finding")
print("=" * 40)
survival_by_class = df.groupby('Pclass')['Survived'].mean()
print("Survival rate by class:")
print(survival_by_class)
print("
First class passengers survived more often — consistent with the")
print("historical record of Titanic evacuations prioritising wealthy passengers.")

---

## EDA Checklist (use on every new dataset)

1. `.shape` — how many rows and columns?
2. `.dtypes` — are types correct? (dates as strings? numeric as object?)
3. `.isnull().sum()` — which columns have missing data, and how much?
4. `.describe()` — do numeric ranges make sense? Any impossible values?
5. `.value_counts()` on categorical columns — any unexpected categories?
6. One or two visualisations to spot patterns and outliers